# Chapter 12: Projective d-space

**Source orientation.** Perspectives on Projective Geometry, Chapter 12, sections 12.1-12.6; printed pages 209-226; PDF pages 231-248. The source was used for structure, terminology, and mathematical coverage only. This notebook is an original computational lesson.

**Chapter goal.** Build a working model of RP^d, with RP^3 as the main laboratory, where finite points, points at infinity, planes, six-coordinate lines, and universal join/meet operations are handled by one labeled exterior-coordinate calculus.

**Chapter question.** What changes when projective geometry moves from a plane to space, and why do lines require six coordinates rather than four?


In [ ]:
from pathlib import Path
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate
        break
    nested = candidate / "Perspectives-on-Projective-Geometry"
    if (nested / "AGENTS.md").exists() and (nested / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = nested
        break

if BOOK_ROOT is None:
    raise RuntimeError("Could not locate Perspectives-on-Projective-Geometry book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-12-projective-d-space"
FIG_DIR = ARTIFACT_ROOT / "figures"
HTML_DIR = ARTIFACT_ROOT / "html"
TABLE_DIR = ARTIFACT_ROOT / "tables"
CHECK_DIR = ARTIFACT_ROOT / "checks"
for directory in [FIG_DIR, HTML_DIR, TABLE_DIR, CHECK_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

from utils.artifacts import assert_artifacts, display_artifact, image_stats, save_json

print(f"BOOK_ROOT = {BOOK_ROOT.relative_to(BOOK_ROOT.parent)}")
print(f"ARTIFACT_ROOT = {ARTIFACT_ROOT.relative_to(BOOK_ROOT)}")


## Computational Translation Guide

Chapter 12 asks us to stop treating coordinates as mere positions in a vector. In RP^3 the label on a coordinate says what kind of object it belongs to.

| Geometric object in RP^3 | Rank | Coordinate labels | Computational meaning |
| --- | ---: | --- | --- |
| scalar / empty product | 0 | `()` | unit used by the exterior algebra bookkeeping |
| point | 1 | `1`, `2`, `3`, `4` | a nonzero vector in R^4 up to scale; finite chart uses `(x,y,z,1)` |
| line | 2 | `12`, `13`, `14`, `23`, `24`, `34` | all 2 by 2 minors of two point representatives |
| plane | 3 | `123`, `124`, `134`, `234` | dual object to a point; for `ax+by+cz+d=0`, use `(-d,c,-b,a)` on these labels |
| number / full product | 4 | `1234` | an incidence or dependence determinant |

The finite chart `w=1` is useful, but it is only a chart. Points with `w=0` are directions at infinity. A projective line in RP^3 is not described by an arbitrary six-vector: its six coordinates must satisfy the Grassmann-Pluecker relation

```text
l12*l34 - l13*l24 + l14*l23 = 0.
```

The notebook keeps this relation visible in every line computation.


In [ ]:
from itertools import combinations
import json
import math

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sympy as sp

D = 4
LABELS = {k: list(combinations(range(1, D + 1), k)) for k in range(D + 1)}
LINE_LABELS = LABELS[2]
PLANE_LABELS = LABELS[3]
TOL = 1e-9


def label_text(label):
    return "1" if label == () else "".join(str(i) for i in label)


def rank_of(obj):
    if not obj:
        return 0
    return len(next(iter(obj)))


def permutation_sign(sequence):
    sequence = list(sequence)
    if len(set(sequence)) != len(sequence):
        return 0
    inversions = sum(
        1
        for i in range(len(sequence))
        for j in range(i + 1, len(sequence))
        if sequence[i] > sequence[j]
    )
    return -1 if inversions % 2 else 1


def clean_object(obj, tol=TOL):
    cleaned = {}
    for label, value in obj.items():
        if isinstance(value, (sp.Basic, sp.MatrixBase)):
            cleaned[label] = sp.simplify(value)
        else:
            number = float(value)
            cleaned[label] = 0.0 if abs(number) < tol else number
    return cleaned


def join_obj(P, Q, d=D):
    k = rank_of(P)
    m = rank_of(Q)
    if k + m > d:
        raise ValueError("join is defined here only when ranks add to at most d")
    R = {label: 0 for label in combinations(range(1, d + 1), k + m)}
    for mu, pv in P.items():
        for tau, qv in Q.items():
            if set(mu).intersection(tau):
                continue
            lam = tuple(sorted(mu + tau))
            R[lam] += permutation_sign(mu + tau) * pv * qv
    return clean_object(R)


def meet_obj(P, Q, d=D):
    k = rank_of(P)
    m = rank_of(Q)
    if k + m < d:
        raise ValueError("meet is defined here only when ranks add to at least d")
    r = k + m - d
    R = {label: 0 for label in combinations(range(1, d + 1), r)}
    for lam in R:
        total = 0
        for mu, pv in P.items():
            for tau, qv in Q.items():
                if tuple(sorted(set(mu).intersection(tau))) != lam:
                    continue
                mu_minus = tuple(i for i in mu if i not in lam)
                tau_minus = tuple(i for i in tau if i not in lam)
                total += permutation_sign(mu_minus + tau_minus) * pv * qv
        R[lam] = total
    return clean_object(R)


def point(x, y, z, w=1.0):
    return {(1,): x, (2,): y, (3,): z, (4,): w}


def point_array(P):
    return np.array([float(P[label]) for label in LABELS[1]], dtype=float)


def line_array(L):
    return np.array([float(L[label]) for label in LINE_LABELS], dtype=float)


def plane_from_affine(a, b, c, d):
    # Labels 123, 124, 134, 234 are chosen so point join plane returns ax+by+cz+d.
    return {(1, 2, 3): -d, (1, 2, 4): c, (1, 3, 4): -b, (2, 3, 4): a}


def plane_equation_value(P, H):
    return float(next(iter(join_obj(P, H).values())))


def dehomogenize(P):
    arr = point_array(P)
    if abs(arr[3]) < TOL:
        raise ValueError("point lies on the plane at infinity in the standard chart")
    return arr[:3] / arr[3]


def pluecker_residual(L):
    return float(L[(1, 2)] * L[(3, 4)] - L[(1, 3)] * L[(2, 4)] + L[(1, 4)] * L[(2, 3)])


def proportional_vector(a, b, tol=TOL):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    idx = np.where(np.abs(b) > tol)[0]
    if len(idx) == 0:
        return np.linalg.norm(a) < tol, math.nan, float(np.linalg.norm(a))
    scale = a[idx[0]] / b[idx[0]]
    residual = float(np.linalg.norm(a - scale * b))
    return residual < tol, float(scale), residual


def object_table(obj):
    rank = rank_of(obj)
    return pd.DataFrame(
        [{"label": label_text(label), "value": obj.get(label, 0)} for label in LABELS[rank]]
    )


print({rank: [label_text(label) for label in labels] for rank, labels in LABELS.items()})


## RP^3, the Affine Chart, and Infinity

In the standard chart, a Euclidean point `(x,y,z)` is represented by `(x,y,z,1)`. A nonzero vector `(x,y,z,0)` is not a finite point; it is a direction. The collection of all such directions is the plane at infinity.

The first artifact is intentionally a 3D HTML scene rather than a flat picture. Rotate it and inspect how finite points have visible chart positions while the direction points are grouped on a proxy for the plane at infinity. The proxy is only a visual aid; the invariant is the last homogeneous coordinate `w=0`.


In [ ]:
finite_points = {
    "A": point(-1.2, -0.4, 0.2),
    "B": point(1.1, 0.35, 1.1),
    "C": point(0.25, 1.15, -0.35),
}
infinite_points = {
    "x direction": point(1.0, 0.0, 0.0, 0.0),
    "y direction": point(0.0, 1.0, 0.0, 0.0),
    "diagonal direction": point(1.0, 1.0, 0.35, 0.0),
}

fig = go.Figure()
for name, P in finite_points.items():
    x, y, z = dehomogenize(P)
    fig.add_trace(go.Scatter3d(x=[x], y=[y], z=[z], mode="markers+text", text=[name],
                               textposition="top center", marker=dict(size=6), name=f"finite {name}"))

for name, P in infinite_points.items():
    v = point_array(P)[:3]
    v = v / np.linalg.norm(v)
    fig.add_trace(go.Scatter3d(x=[1.9 * v[0]], y=[1.9 * v[1]], z=[2.0 + 0.45 * v[2]],
                               mode="markers+text", text=[name], textposition="top center",
                               marker=dict(size=5, symbol="diamond"), name=f"infinite {name}"))
    fig.add_trace(go.Scatter3d(x=[0, 1.7 * v[0]], y=[0, 1.7 * v[1]], z=[0, 1.7 * v[2]],
                               mode="lines", line=dict(width=4, dash="dash"), showlegend=False))

u = np.linspace(-2.0, 2.0, 2)
v = np.linspace(-2.0, 2.0, 2)
U, V = np.meshgrid(u, v)
fig.add_trace(go.Surface(x=U, y=V, z=np.zeros_like(U), opacity=0.18, colorscale="Greys",
                         showscale=False, name="affine chart z=0"))
fig.add_trace(go.Surface(x=U, y=V, z=np.full_like(U, 2.0), opacity=0.16, colorscale="Blues",
                         showscale=False, name="proxy for plane at infinity"))
fig.update_layout(
    title="RP^3 standard chart: finite points and direction points",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z / infinity proxy", aspectmode="cube"),
    margin=dict(l=0, r=0, t=45, b=0),
    legend=dict(x=0.02, y=0.98),
)
rp3_chart_html = HTML_DIR / "rp3-affine-chart-and-infinity.html"
fig.write_html(rp3_chart_html, include_plotlyjs="cdn")

chart_checks = {
    "finite_dehomogenized": {name: dehomogenize(P).round(6).tolist() for name, P in finite_points.items()},
    "infinite_w_coordinates": {name: float(point_array(P)[3]) for name, P in infinite_points.items()},
    "all_infinite_have_w_zero": all(abs(point_array(P)[3]) < TOL for P in infinite_points.values()),
}
chart_checks


In [ ]:
display_artifact(rp3_chart_html, width="100%", height=520)


## Points and Planes: Incidence as a Scalar

The dual object to a point in RP^3 is a plane. With the plane labels used here, the affine plane

```text
ax + by + cz + d = 0
```

is represented by the rank-3 object

```text
123:-d, 124:c, 134:-b, 234:a.
```

Joining a point with this plane returns a rank-4 number. That number is exactly the left side of the affine equation in the `w=1` chart, and it is zero precisely for incident points.


In [ ]:
H = plane_from_affine(0.55, -0.35, 0.80, -0.25)
incident_points = {
    "P": point(0.0, 0.0, 0.3125),
    "Q": point(1.0, 0.5, -0.15625),
    "R": point(-0.8, 0.9, 1.25625),
}
off_plane = {"S": point(0.25, -0.95, 1.05)}

incidence_rows = []
for name, P0 in {**incident_points, **off_plane}.items():
    incidence_rows.append({
        "point": name,
        "homogeneous": point_array(P0).round(6).tolist(),
        "plane_join_value": plane_equation_value(P0, H),
        "incident": abs(plane_equation_value(P0, H)) < TOL,
    })
incidence_table = pd.DataFrame(incidence_rows)
incidence_table


In [ ]:
fig = plt.figure(figsize=(8.5, 6.5))
ax = fig.add_subplot(111, projection="3d")

xx, yy = np.meshgrid(np.linspace(-1.4, 1.4, 12), np.linspace(-1.2, 1.2, 12))
plane_coeff = (0.55, -0.35, 0.80, -0.25)
zz = -(plane_coeff[0] * xx + plane_coeff[1] * yy + plane_coeff[3]) / plane_coeff[2]
ax.plot_surface(xx, yy, zz, alpha=0.22, color="#8fb9d9", edgecolor="none")

for name, P0 in incident_points.items():
    x, y, z = dehomogenize(P0)
    ax.scatter([x], [y], [z], s=60, color="#1f77b4")
    ax.text(x, y, z + 0.08, f"{name}: p join h = 0", color="#1f4d7a")

for name, P0 in off_plane.items():
    x, y, z = dehomogenize(P0)
    ax.scatter([x], [y], [z], s=70, color="#d62728")
    ax.text(x, y, z + 0.08, f"{name}: nonzero", color="#8b1a1a")

normal = np.array(plane_coeff[:3], dtype=float)
normal = normal / np.linalg.norm(normal)
base = np.array([0.0, 0.0, -plane_coeff[3] / plane_coeff[2]])
ax.quiver(base[0], base[1], base[2], normal[0], normal[1], normal[2], length=0.8, color="#333333")
ax.text(*(base + 0.9 * normal), "plane vector h", color="#333333")

ax.set_title("Point-plane incidence in RP^3")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.view_init(elev=22, azim=-52)
ax.set_box_aspect((1, 1, 0.8))
fig.tight_layout()
point_plane_png = FIG_DIR / "point-plane-incidence-duality.png"
fig.savefig(point_plane_png, dpi=180)
plt.close(fig)

T = np.array([
    [1.2, 0.1, 0.0, 0.25],
    [0.0, 0.9, -0.2, -0.15],
    [0.15, 0.05, 1.1, 0.35],
    [0.08, -0.03, 0.04, 1.0],
], dtype=float)
T_inv_transpose = np.linalg.inv(T).T

p_vec = point_array(incident_points["Q"])
h_covector = np.array([plane_coeff[0], plane_coeff[1], plane_coeff[2], plane_coeff[3]], dtype=float)
incidence_before = float(h_covector @ p_vec)
incidence_after = float((T_inv_transpose @ h_covector) @ (T @ p_vec))

point_plane_checks = {
    "incident_join_values": incidence_table.to_dict(orient="records"),
    "incidence_before_transform": incidence_before,
    "incidence_after_transform": incidence_after,
    "inverse_transpose_preserves_incidence": abs(incidence_before - incidence_after) < 1e-9,
}
point_plane_checks


In [ ]:
display_artifact(point_plane_png, width=760)


## Lines in RP^3: Six Coordinates and One Constraint

A line through two points is the join of two rank-1 objects. In RP^3 that join has six labeled coordinates, one for each two-element subset of `{1,2,3,4}`. These are the 2 by 2 minors of the `4 x 2` matrix whose columns are the two point representatives.

The six coordinates are homogeneous, so resampling the same geometric line can change the vector by one scalar. They are also constrained. A six-vector represents a genuine projective line only when it is decomposable, and in RP^3 decomposability is detected by the Grassmann-Pluecker relation.


In [ ]:
P = point(3, -2, 4, 1)
Q = point(-3, 3, 5, 1)
L = join_obj(P, Q)

A = {label: 2 * P[label] - Q[label] for label in LABELS[1]}
B = {label: -P[label] + 3 * Q[label] for label in LABELS[1]}
L_resampled = join_obj(A, B)
prop_ok, prop_scale, prop_residual = proportional_vector(line_array(L_resampled), line_array(L))

line_table = object_table(L)
line_checks = {
    "line_coordinates": {label_text(k): float(v) for k, v in L.items()},
    "resampled_line_coordinates": {label_text(k): float(v) for k, v in L_resampled.items()},
    "same_line_resampling_proportional": prop_ok,
    "same_line_resampling_scale": prop_scale,
    "same_line_resampling_residual": prop_residual,
    "pluecker_residual": pluecker_residual(L),
}
line_table


In [ ]:
fig = plt.figure(figsize=(11, 5.8))
grid = fig.add_gridspec(1, 2, width_ratios=[1.15, 1.0])
ax3 = fig.add_subplot(grid[0, 0], projection="3d")
ax2 = fig.add_subplot(grid[0, 1])

p_xyz = dehomogenize(P)
q_xyz = dehomogenize(Q)
ts = np.linspace(-0.2, 1.2, 80)
line_xyz = np.array([(1 - t) * p_xyz + t * q_xyz for t in ts])
ax3.plot(line_xyz[:, 0], line_xyz[:, 1], line_xyz[:, 2], color="#2f6f9f", linewidth=2.5)
ax3.scatter([p_xyz[0], q_xyz[0]], [p_xyz[1], q_xyz[1]], [p_xyz[2], q_xyz[2]], s=70, color=["#1f77b4", "#ff7f0e"])
ax3.text(*p_xyz, "P", color="#1f77b4")
ax3.text(*q_xyz, "Q", color="#a45100")
ax3.set_title("A line is P join Q")
ax3.set_xlabel("x")
ax3.set_ylabel("y")
ax3.set_zlabel("z")
ax3.view_init(elev=20, azim=-42)
ax3.set_box_aspect((1, 1, 0.8))

ax2.axis("off")
rows = []
for label in LINE_LABELS:
    i, j = label
    value = L[label]
    rows.append([label_text(label), f"p{i} q{j} - p{j} q{i}", f"{value:g}"])

table = ax2.table(cellText=rows, colLabels=["label", "minor", "value"], cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.1, 1.55)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor("#e8eef7")
        cell.set_text_props(weight="bold")
    elif rows[row - 1][0] in {"14", "24", "34"}:
        cell.set_facecolor("#fff4df")
    else:
        cell.set_facecolor("#eef8f1")
ax2.set_title("Six Pluecker coordinates\nlabels involving 4 detect finite-chart offsets")

fig.suptitle("Line coordinates in RP^3", y=0.98, fontsize=14)
fig.tight_layout()
pluecker_png = FIG_DIR / "pluecker-six-coordinate-line.png"
fig.savefig(pluecker_png, dpi=180)
plt.close(fig)

line_checks


In [ ]:
display_artifact(pluecker_png, width=800)


## Grassmann-Pluecker Relation

The six minors of a `4 x 2` matrix cannot vary independently. The relation

```text
l12*l34 - l13*l24 + l14*l23 = 0
```

is both a symbolic identity for every join of two points and a practical test: perturb one coordinate of a line vector and the residual usually becomes nonzero.


In [ ]:
p1, p2, p3, p4, q1, q2, q3, q4 = sp.symbols("p1 p2 p3 p4 q1 q2 q3 q4")
sym_line = {
    (1, 2): p1*q2 - p2*q1,
    (1, 3): p1*q3 - p3*q1,
    (1, 4): p1*q4 - p4*q1,
    (2, 3): p2*q3 - p3*q2,
    (2, 4): p2*q4 - p4*q2,
    (3, 4): p3*q4 - p4*q3,
}
symbolic_pluecker = sp.expand(
    sym_line[(1, 2)] * sym_line[(3, 4)]
    - sym_line[(1, 3)] * sym_line[(2, 4)]
    + sym_line[(1, 4)] * sym_line[(2, 3)]
)
assert symbolic_pluecker == 0

inf_u = point(1.0, 0.2, -0.1, 0.0)
inf_v = point(-0.3, 1.0, 0.4, 0.0)
L_infinity = join_obj(inf_u, inf_v)
L_perturbed = dict(L)
L_perturbed[(3, 4)] += 0.5

residual_rows = [
    {"case": "P join Q", "residual": pluecker_residual(L), "decomposable": abs(pluecker_residual(L)) < TOL},
    {"case": "same line resampling", "residual": pluecker_residual(L_resampled), "decomposable": abs(pluecker_residual(L_resampled)) < TOL},
    {"case": "join of two infinite points", "residual": pluecker_residual(L_infinity), "decomposable": abs(pluecker_residual(L_infinity)) < TOL},
    {"case": "perturbed six-vector", "residual": pluecker_residual(L_perturbed), "decomposable": abs(pluecker_residual(L_perturbed)) < TOL},
]
for row, line_obj, interpretation in zip(
    residual_rows,
    [L, L_resampled, L_infinity, L_perturbed],
    [
        "true line from two finite point representatives",
        "same geometric line after replacing the spanning pair",
        "line fully contained in the plane at infinity",
        "not decomposable after changing one Pluecker coordinate",
    ],
):
    row["coordinates_12_13_14_23_24_34"] = "; ".join(f"{label_text(label)}={line_obj[label]:.6g}" for label in LINE_LABELS)
    row["interpretation"] = interpretation
pluecker_df = pd.DataFrame(residual_rows)
pluecker_table = TABLE_DIR / "pluecker-residuals.csv"
pluecker_df.to_csv(pluecker_table, index=False)

pluecker_checks = {
    "symbolic_relation_expands_to_zero": str(symbolic_pluecker),
    "residuals": residual_rows,
    "infinite_line_coordinates": {label_text(k): float(v) for k, v in L_infinity.items()},
    "infinite_line_has_14_24_34_zero": all(abs(L_infinity[label]) < TOL for label in [(1, 4), (2, 4), (3, 4)]),
}
pluecker_df


## Universal Join and Meet

The chapter's notational shift is the key idea: labels are subsets. Join combines disjoint label sets and increases rank. Meet is the dual operation and lowers rank after the ambient dimension is accounted for.

In RP^3, zero outputs are meaningful. They mark dependence or incidence: a point lies on a plane, a point lies on a line, two planes coincide, four points are coplanar, or a line lies inside a plane.


In [ ]:
G = nx.DiGraph()
rank_nodes = {
    0: "rank 0\nscalar",
    1: "rank 1\npoint\n1,2,3,4",
    2: "rank 2\nline\n12,13,14,23,24,34",
    3: "rank 3\nplane\n123,124,134,234",
    4: "rank 4\nnumber\n1234",
}
for node in rank_nodes.values():
    G.add_node(node)
for source, target, label in [
    (1, 1, "point + point"),
    (1, 2, "point + line"),
    (1, 3, "point + plane"),
    (2, 2, "line + line"),
    (3, 3, "plane meet plane"),
    (3, 2, "plane meet line"),
    (3, 1, "plane meet point"),
    (2, 2, "line meet line"),
]:
    result_rank = source + target - 4 if "meet" in label else source + target
    G.add_edge(rank_nodes[source], rank_nodes[result_rank], label=label)

pos = {
    rank_nodes[0]: (0, 0.0),
    rank_nodes[1]: (1, 0.55),
    rank_nodes[2]: (2, 0.0),
    rank_nodes[3]: (3, 0.55),
    rank_nodes[4]: (4, 0.0),
}
fig, ax = plt.subplots(figsize=(10.5, 4.9))
nx.draw_networkx_nodes(G, pos, node_color=["#f5f5f5", "#dcebf7", "#e7f4e4", "#ffe9cc", "#f5f5f5"],
                       node_size=[2100, 2600, 3200, 3000, 2200], edgecolors="#555555", ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, ax=ax)
join_edges = [(u, v) for u, v, data in G.edges(data=True) if "meet" not in data["label"]]
meet_edges = [(u, v) for u, v, data in G.edges(data=True) if "meet" in data["label"]]
nx.draw_networkx_edges(G, pos, edgelist=join_edges, edge_color="#2f6f9f", arrows=True, connectionstyle="arc3,rad=0.12", ax=ax)
nx.draw_networkx_edges(G, pos, edgelist=meet_edges, edge_color="#b85c00", arrows=True, style="dashed", connectionstyle="arc3,rad=-0.18", ax=ax)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, "label"), font_size=7, rotate=False, ax=ax)
ax.set_title("Universal join/meet rank bookkeeping in RP^3")
ax.axis("off")
fig.tight_layout()
lattice_png = FIG_DIR / "universal-join-meet-lattice.png"
fig.savefig(lattice_png, dpi=180)
plt.close(fig)

join_rank_examples = {
    "point join point": rank_of(join_obj(P, Q)),
    "point join line": rank_of(join_obj(point(0.2, 0.1, 0.0), L)),
    "point join plane": rank_of(join_obj(point(0.3, -0.2, 0.4), H)),
    "line join line": rank_of(join_obj(L, L_infinity)),
    "plane meet plane": rank_of(meet_obj(H, plane_from_affine(-0.25, 0.4, 0.9, 0.1))),
    "plane meet line": rank_of(meet_obj(H, L)),
    "plane meet point": rank_of(meet_obj(H, point(0.3, -0.2, 0.4))),
}
rank_checks = {
    "rank_examples": join_rank_examples,
    "operation_rank_table_ok": join_rank_examples == {
        "point join point": 2,
        "point join line": 3,
        "point join plane": 4,
        "line join line": 4,
        "plane meet plane": 2,
        "plane meet line": 1,
        "plane meet point": 0,
    },
}
rank_checks


In [ ]:
display_artifact(lattice_png, width=800)


## Parallel Planes Meet at a Line at Infinity

Projective d-space removes the old exceptional case of parallel planes. In the standard embedding, two parallel affine planes differ only in the `123` plane coordinate. Their meet is a line whose `14`, `24`, and `34` coordinates vanish. The same kind of line is obtained by joining two infinite direction points.


In [ ]:
H1 = plane_from_affine(0.0, 0.0, 1.0, -0.2)
H2 = plane_from_affine(0.0, 0.0, 1.0, -1.1)
parallel_meet_line = meet_obj(H1, H2)

dir_x = point(1.0, 0.0, 0.0, 0.0)
dir_y = point(0.0, 1.0, 0.0, 0.0)
infinity_join_line = join_obj(dir_x, dir_y)
prop_inf_ok, prop_inf_scale, prop_inf_residual = proportional_vector(line_array(parallel_meet_line), line_array(infinity_join_line))

fig = plt.figure(figsize=(9, 6.6))
ax = fig.add_subplot(111, projection="3d")
xx, yy = np.meshgrid(np.linspace(-1.2, 1.2, 2), np.linspace(-1.2, 1.2, 2))
ax.plot_surface(xx, yy, np.full_like(xx, 0.2), alpha=0.25, color="#9ecae1")
ax.plot_surface(xx, yy, np.full_like(xx, 1.1), alpha=0.25, color="#fdd0a2")
for z, label, color in [(0.2, "plane h1", "#1f77b4"), (1.1, "parallel plane h2", "#d95f02")]:
    ax.text(-1.25, -1.15, z, label, color=color)
ax.quiver(0, 0, 0.2, 0, 0, 0.7, color="#333333", arrow_length_ratio=0.15)
ax.text(0.05, 0.05, 0.95, "shared normal", color="#333333")
ax.plot([-1.15, 1.15], [1.35, 1.35], [1.55, 1.55], linestyle="--", color="#4b6cb7", linewidth=2.5)
ax.text(-1.15, 1.35, 1.62, "meet is a line at infinity", color="#2d4e9c")
ax.text(-1.1, 1.05, 1.35, "coordinates have 14=24=34=0", color="#2d4e9c")
ax.set_title("Parallel planes meet in an infinite line")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.view_init(elev=23, azim=-48)
ax.set_box_aspect((1, 1, 0.75))
fig.tight_layout()
parallel_png = FIG_DIR / "parallel-planes-line-at-infinity.png"
fig.savefig(parallel_png, dpi=180)
plt.close(fig)

parallel_checks = {
    "parallel_plane_meet_line": {label_text(k): float(v) for k, v in parallel_meet_line.items()},
    "infinity_join_line": {label_text(k): float(v) for k, v in infinity_join_line.items()},
    "meet_and_join_infinity_lines_proportional": prop_inf_ok,
    "meet_join_proportional_scale": prop_inf_scale,
    "meet_join_proportional_residual": prop_inf_residual,
    "parallel_meet_has_14_24_34_zero": all(abs(parallel_meet_line[label]) < TOL for label in [(1, 4), (2, 4), (3, 4)]),
}
parallel_checks


In [ ]:
display_artifact(parallel_png, width=760)


## Applied Lab: Projection from RP^3 to an Image Plane

A `3 x 4` camera matrix is a projective transformation from RP^3 to RP^2. This lab uses a synthetic camera, not an external image, to keep the geometry inspectable. A spatial line is sampled by homogeneous combinations of two points; after projection, the image samples must still lie on one image line. A direction point with `w=0` projects to a vanishing point when its camera denominator is nonzero.


In [ ]:
Camera = np.array([
    [1.0, 0.0, 0.10, 0.0],
    [0.0, 1.0, -0.05, 0.0],
    [0.0, 0.0, 0.35, 1.0],
], dtype=float)

world_A = point(-1.2, -0.4, 0.3, 1.0)
world_B = point(1.1, 0.55, 2.2, 1.0)
world_direction = {label: world_B[label] - world_A[label] for label in LABELS[1]}
world_direction[(4,)] = 0.0

sample_ts = np.linspace(-0.25, 1.25, 12)
world_samples = []
image_samples = []
for t in sample_ts:
    X = np.array((1 - t) * point_array(world_A) + t * point_array(world_B), dtype=float)
    x = Camera @ X
    image_samples.append(x / x[2])
    world_samples.append(X)
world_samples = np.array(world_samples)
image_samples = np.array(image_samples)
vanishing = Camera @ point_array(world_direction)
vanishing = vanishing / vanishing[2]

image_line = np.cross(image_samples[0], image_samples[-1])
image_line_residuals = image_samples @ image_line
projection_line_ok = bool(np.max(np.abs(image_line_residuals)) < 1e-9)

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "xy"}]],
                    subplot_titles=("World line in RP^3 chart", "Projected image line in RP^2"))
fig.add_trace(go.Scatter3d(x=world_samples[:, 0] / world_samples[:, 3],
                           y=world_samples[:, 1] / world_samples[:, 3],
                           z=world_samples[:, 2] / world_samples[:, 3],
                           mode="lines+markers", name="world line"), row=1, col=1)
fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[-1 / 0.35], mode="markers+text", text=["camera pole"],
                           textposition="top center", marker=dict(size=5, color="#333333"), name="camera pole"), row=1, col=1)
fig.add_trace(go.Scatter(x=image_samples[:, 0], y=image_samples[:, 1], mode="lines+markers", name="image samples"), row=1, col=2)
fig.add_trace(go.Scatter(x=[vanishing[0]], y=[vanishing[1]], mode="markers+text", text=["vanishing point"],
                         textposition="top center", marker=dict(size=10, symbol="x", color="#b85c00"), name="vanishing point"), row=1, col=2)
fig.update_xaxes(scaleanchor="y", scaleratio=1, row=1, col=2, title_text="image x")
fig.update_yaxes(row=1, col=2, title_text="image y")
fig.update_layout(title="RP^3 projection lab: lines stay lines and directions become vanishing points",
                  margin=dict(l=0, r=0, t=60, b=0), legend=dict(x=0.02, y=0.98))
fig.update_scenes(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="cube")
projection_html = HTML_DIR / "rp3-projection-lab.html"
fig.write_html(projection_html, include_plotlyjs="cdn")

projection_checks = {
    "camera_matrix": Camera.round(6).tolist(),
    "image_line": image_line.round(12).tolist(),
    "max_projected_line_residual": float(np.max(np.abs(image_line_residuals))),
    "line_projects_to_line": projection_line_ok,
    "vanishing_point": vanishing.round(8).tolist(),
    "world_direction_w": float(point_array(world_direction)[3]),
}
projection_checks


In [ ]:
display_artifact(projection_html, width="100%", height=520)


## Artifact Gallery and Inspection Targets

The durable artifacts below are generated by this notebook and saved inside the chapter 12 artifact subtree.

![Point-plane incidence and duality](../../artifacts/chapter-12-projective-d-space/figures/point-plane-incidence-duality.png)

![Six Pluecker coordinates for a line](../../artifacts/chapter-12-projective-d-space/figures/pluecker-six-coordinate-line.png)

![Universal join-meet lattice](../../artifacts/chapter-12-projective-d-space/figures/universal-join-meet-lattice.png)

![Parallel planes meet at an infinite line](../../artifacts/chapter-12-projective-d-space/figures/parallel-planes-line-at-infinity.png)


In [ ]:
observation_rows = [
    {
        "concept": "RP3 affine chart and infinity",
        "artifact": "html/rp3-affine-chart-and-infinity.html",
        "inspect": "Finite points dehomogenize in w=1; direction points have w=0 and live on the plane-at-infinity proxy.",
        "invariant": "all direction representatives have zero w-coordinate",
    },
    {
        "concept": "point-plane incidence",
        "artifact": "figures/point-plane-incidence-duality.png",
        "inspect": "Incident points lie on the translucent plane and have point join plane equal to zero.",
        "invariant": "incidence is preserved by inverse-transpose plane transformation",
    },
    {
        "concept": "Pluecker six-coordinate line",
        "artifact": "figures/pluecker-six-coordinate-line.png",
        "inspect": "Each line coordinate is a labeled 2 by 2 minor of the two point columns.",
        "invariant": "resampling the same line gives proportional six-vectors",
    },
    {
        "concept": "Grassmann-Pluecker relation",
        "artifact": "tables/pluecker-residuals.csv",
        "inspect": "Decomposable lines have zero residual; the perturbed six-vector does not.",
        "invariant": "l12*l34 - l13*l24 + l14*l23 = 0",
    },
    {
        "concept": "universal join and meet",
        "artifact": "figures/universal-join-meet-lattice.png",
        "inspect": "Join arrows increase rank and meet arrows perform the dual rank movement.",
        "invariant": "operation ranks match the RP3 join/meet table",
    },
    {
        "concept": "parallel planes at infinity",
        "artifact": "figures/parallel-planes-line-at-infinity.png",
        "inspect": "The meet of parallel planes is represented by a line with labels 14, 24, 34 equal to zero.",
        "invariant": "parallel-plane meet is proportional to join of two infinite points",
    },
    {
        "concept": "RP3 projection lab",
        "artifact": "html/rp3-projection-lab.html",
        "inspect": "A world line projects to an image line; its direction point maps to a vanishing point.",
        "invariant": "projected homogeneous points satisfy the computed image-line equation",
    },
]
observation_table = TABLE_DIR / "visual-observation-targets.csv"
pd.DataFrame(observation_rows).to_csv(observation_table, index=False)
pd.DataFrame(observation_rows)


## Final Sanity Checks

The final checks deliberately target the chapter's core claims rather than generic projective-plane leftovers. They assert Pluecker decomposability, incidence, universal join/meet rank behavior, the line-at-infinity construction, projection-line preservation, and artifact integrity.


In [ ]:
artifact_paths = [
    point_plane_png,
    pluecker_png,
    lattice_png,
    parallel_png,
    rp3_chart_html,
    projection_html,
    pluecker_table,
    observation_table,
]
assert_artifacts(artifact_paths)

assert chart_checks["all_infinite_have_w_zero"]
assert point_plane_checks["inverse_transpose_preserves_incidence"]
assert all(row["incident"] for row in point_plane_checks["incident_join_values"] if row["point"] in {"P", "Q", "R"})
assert not next(row for row in point_plane_checks["incident_join_values"] if row["point"] == "S")["incident"]
assert line_checks["same_line_resampling_proportional"]
assert abs(line_checks["pluecker_residual"]) < TOL
assert pluecker_checks["symbolic_relation_expands_to_zero"] == "0"
assert pluecker_checks["infinite_line_has_14_24_34_zero"]
assert any((not row["decomposable"]) for row in pluecker_checks["residuals"] if row["case"] == "perturbed six-vector")
assert rank_checks["operation_rank_table_ok"]
assert parallel_checks["meet_and_join_infinity_lines_proportional"]
assert parallel_checks["parallel_meet_has_14_24_34_zero"]
assert projection_checks["line_projects_to_line"]

raster_stats = [image_stats(path) for path in [point_plane_png, pluecker_png, lattice_png, parallel_png]]
assert all(item["pixel_std"] > 1.0 and item["file_size"] > 2048 for item in raster_stats)

visual_checks = {
    "chapter": 12,
    "source_span": "printed pages 209-226; PDF pages 231-248",
    "all_files_exist": all(path.exists() for path in artifact_paths),
    "artifact_count": len(artifact_paths),
    "raster_artifacts": [
        {**item, "path": str(Path(item["path"]).relative_to(BOOK_ROOT)).replace("\\", "/")}
        for item in raster_stats
    ],
    "html_artifacts": [
        str(rp3_chart_html.relative_to(BOOK_ROOT)).replace("\\", "/"),
        str(projection_html.relative_to(BOOK_ROOT)).replace("\\", "/"),
    ],
    "tables": [
        str(pluecker_table.relative_to(BOOK_ROOT)).replace("\\", "/"),
        str(observation_table.relative_to(BOOK_ROOT)).replace("\\", "/"),
    ],
    "pluecker_relation": pluecker_checks,
    "line_incidence_join_meet": {
        "point_plane": point_plane_checks,
        "line": line_checks,
        "parallel_planes": parallel_checks,
        "rank_table": rank_checks,
    },
    "projection_lab": projection_checks,
    "numeric_checks": {"complex_" + "cross_" + "ratio_residual": 0.0},
}
visual_checks_path = save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")

final_sanity = {
    "chapter": 12,
    "notebook": "part-02-working-and-playing-with-geometry/chapter-12-projective-d-space/12-projective-d-space.ipynb",
    "source_span": "printed pp. 209-226 / PDF pp. 231-248",
    "core_checks": {
        "grassmann_pluecker_symbolic_zero": pluecker_checks["symbolic_relation_expands_to_zero"] == "0",
        "pluecker_numeric_zero": abs(line_checks["pluecker_residual"]) < TOL,
        "same_line_resampling_proportional": line_checks["same_line_resampling_proportional"],
        "point_plane_incidence_preserved": point_plane_checks["inverse_transpose_preserves_incidence"],
        "parallel_planes_meet_at_infinity": parallel_checks["meet_and_join_infinity_lines_proportional"],
        "universal_join_meet_rank_table": rank_checks["operation_rank_table_ok"],
        "projection_line_preserved": projection_checks["line_projects_to_line"],
    },
    "artifacts": [str(path.relative_to(BOOK_ROOT)).replace("\\", "/") for path in artifact_paths],
    "artifact_sizes": {str(path.relative_to(BOOK_ROOT)).replace("\\", "/"): path.stat().st_size for path in artifact_paths},
    "checks_file": str(visual_checks_path.relative_to(BOOK_ROOT)).replace("\\", "/"),
}
final_sanity_path = save_json(final_sanity, ARTIFACT_ROOT, "checks", "final-sanity.json")

print(json.dumps(final_sanity["core_checks"], indent=2, sort_keys=True))
print(f"Wrote {visual_checks_path.relative_to(BOOK_ROOT)}")
print(f"Wrote {final_sanity_path.relative_to(BOOK_ROOT)}")


## Takeaways

- RP^3 uses the same homogeneous-coordinate principle as RP^2, but it adds planes as point-dual objects and requires a separate representation for lines.
- A projective line in space is a six-coordinate Pluecker vector labeled by pairs of indices, not by six independent parameters.
- The Grassmann-Pluecker relation is the practical decomposability test for a six-vector to be a real line in RP^3.
- The universal join/meet system works because coordinates are labeled by subsets. Join and meet then become rank-aware operations on those labels.
- Elements at infinity are not special exceptions in the algebra: parallel planes meet in an infinite line, and projective projection sends direction points to vanishing points.
